# 🦆 Microduck RL — Train SPIN (Xoay tại chỗ) trên Google Colab T4

Model **spin** là model còn thiếu duy nhất của bộ multi mini model (train trên
robot rollers, target 8000 iters). Notebook này được thiết kế cho **phiên Colab
5 giờ 30 phút, GPU T4 (15 GB VRAM)** với 3 nguyên tắc:

1. **Smoke test bắt buộc** trước train dài (quy tắc AGENTS.md).
2. **Train theo chunk 500 iters + resume** — Colab ngắt giữa chừng chỉ mất
   tối đa 1 chunk; chạy lại notebook là tự tiếp tục từ checkpoint trên Drive.
3. **Backup lên Google Drive sau mỗi chunk** — logs + checkpoint + ONNX +
   GIF digital twin đều nằm trên Drive sau khi chạy xong, tải về được cả khi
   phiên đã tắt.

Ngân sách thời gian: để 30 phút dự phòng cho export ONNX + digital twin +
backup, phần còn lại (~5 h) dành cho train. Nếu T4 đủ nhanh sẽ trọn 8000
iters, không thì dừng an toàn ở mốc gần nhất (≥90% vẫn tính READY trong matrix).


### Bước 1: Kiểm tra GPU (T4) & Lấy mã nguồn (nhánh RTX)
> Nếu repo là riêng tư: đổi remote sang HTTPS kèm Personal Access Token,
> hoặc upload zip mã nguồn vào `/content/Microduck` rồi chạy tiếp từ Bước 2.


In [ ]:
!nvidia-smi

import os
if not os.path.exists("Microduck"):
    !git clone -b RTX https://github.com/rootkind35-del/Microduck.git
%cd Microduck
!git log --oneline -3
!git branch --show-current

### Bước 2: Cài đặt `uv` và đồng bộ môi trường *(fresh uv sync là ground truth)*

In [ ]:
!pip install -q uv
!uv sync

### Bước 3: Mount Google Drive — nơi lưu checkpoint + model + logs

Nếu đã train lần trước (phiên trước bị ngắt), checkpoint trên Drive sẽ được
khôi phục về `logs/` và train **tự tiếp tục** thay vì bắt đầu lại.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/Microduck_RTX/spin'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive backup dir:', DRIVE_DIR)

# Khôi phục checkpoint từ các lần chạy trước (nếu có)
import glob, shutil
prev_ckpts = sorted(glob.glob(f'{DRIVE_DIR}/runs/*/model_*.pt'))
if prev_ckpts:
    latest_run = os.path.dirname(prev_ckpts[-1])
    dest = os.path.join('logs/rsl_rl/spin', os.path.basename(latest_run))
    if not os.path.exists(dest):
        shutil.copytree(latest_run, dest)
    print(f'↩ Đã khôi phục {len(prev_ckpts)} checkpoint từ: {latest_run}')
else:
    print('Không có checkpoint cũ trên Drive — train mới từ đầu.')

### Bước 4: SMOKE TEST — 64 envs × 5 iters (BẮT BUỘC trước train dài)

Xác nhận CUDA, MuJoCo Warp, BAM actuator và toàn bộ reward hoạt động, obs
đúng 61D. Chạy ~2 phút, tiết kiệm hàng giờ debug nếu cấu hình có lỗi.

In [ ]:
%env WANDB_MODE=offline
!uv run train Mjlab-Spin-Flat-MicroDuck --env.scene.num-envs 64 --agent.max_iterations 5

### Bước 5: Huấn luyện chính thức — theo ngân sách 5 h 30 của phiên

Mỗi vòng: train tiếp **500 iters** (resume từ checkpoint mới nhất) → chép
checkpoint + logs lên Drive. Vòng lặp tự dừng khi (a) trọn 8000 iters, hoặc
(b) còn 30 phút cuối phiên (giành cho export + twin + backup). Colab ngắt
giữa chừng? Chỉ cần chạy lại notebook từ Bước 1 — tự resume.

In [ ]:
import glob, os, shutil, subprocess, time

SESSION_BUDGET_MIN = 330   # 5 h 30 của phiên Colab
RESERVE_MIN = 30           # dự phòng export ONNX + digital twin + backup
CHUNK_ITERS = 500
TARGET_ITERS = 8000
NUM_ENVS = 1024            # cấu hình chuẩn đã chứng minh (RTX 4060 local)
START = time.time()

def elapsed_min():
    return (time.time() - START) / 60

def latest_ckpt():
    cks = sorted(glob.glob('logs/rsl_rl/spin/*/model_*.pt'), key=os.path.getmtime)
    return cks[-1] if cks else None

def latest_iter():
    ck = latest_ckpt()
    return int(ck.split('model_')[-1].split('.')[0]) if ck else 0

def sync_to_drive():
    """Chép run mới nhất (checkpoint + params + tfevents) lên Drive."""
    runs = sorted(glob.glob('logs/rsl_rl/spin/*/'), key=os.path.getmtime)
    if not runs:
        return
    run = runs[-1].rstrip('/')
    dest = f'{DRIVE_DIR}/runs/{os.path.basename(run)}'
    os.makedirs(dest, exist_ok=True)
    for pattern in ('model_*.pt', '*.onnx', 'params/*', '*.tfevents.*'):
        for f in glob.glob(os.path.join(run, pattern)):
            shutil.copy2(f, dest)
    # Giữ tối đa 3 checkpoint mới nhất trên Drive cho gọn
    cks = sorted(glob.glob(f'{dest}/model_*.pt'),
                 key=lambda p: int(p.split('model_')[-1].split('.')[0]))
    for old in cks[:-3]:
        os.remove(old)

# Dọn run smoke (≤ 50 iters) để không resume nhầm checkpoint thử nghiệm
for run in glob.glob('logs/rsl_rl/spin/*/'):
    cks = [int(p.split('model_')[-1].split('.')[0])
           for p in glob.glob(run + 'model_*.pt')]
    if cks and max(cks) <= 10:
        shutil.rmtree(run)
        print('🧹 Đã dọn run smoke:', run)

while True:
    it = latest_iter()
    if it >= TARGET_ITERS:
        print(f'✓ Hoàn thành {it}/{TARGET_ITERS} iters!')
        break
    remaining = SESSION_BUDGET_MIN - elapsed_min()
    if remaining <= RESERVE_MIN:
        print(f'⏱ Hết ngân sách train (còn {remaining:.0f} phút) — dừng ở {it} iters, chuyển sang export.')
        break
    nxt = min(it + CHUNK_ITERS, TARGET_ITERS)
    cmd = ['uv', 'run', 'train', 'Mjlab-Spin-Flat-MicroDuck',
           '--env.scene.num-envs', str(NUM_ENVS),
           '--agent.max_iterations', str(nxt)]
    ck = latest_ckpt()
    if ck:
        cmd += ['--agent.resume', 'True', f'--agent.load-checkpoint={ck}']
    print(f'▶ Chunk: {it} → {nxt} iters (đã chạy {elapsed_min():.0f} phút, còn {remaining:.0f} phút)')
    rc = subprocess.run(cmd).returncode
    sync_to_drive()
    if rc != 0:
        print(f'✗ Train rc={rc} — kiểm tra log phía trên, sửa rồi chạy lại cell (tự resume).')
        break

print(f'Kết thúc train: {latest_iter()} iters, tổng {elapsed_min():.0f} phút')

### Bước 6: Xuất ONNX (bake normalizer) + Digital Twin kiểm chứng

Export đúng đường bắt buộc (`scripts/export.py` — normalizer baked vào graph,
không bao giờ convert tay checkpoint), rồi chạy digital twin CPU MuJoCo ngay
trên Colab: GIF rollout + chỉ số (góc xoay 1 chu kỳ ≈ 361°).

In [ ]:
%%bash
set -e -o pipefail
CKPT=$(ls -t logs/rsl_rl/spin/*/model_*.pt | head -1)
RUN_DIR=$(dirname "$CKPT")
echo "Export từ: $CKPT"
uv run scripts/export.py Mjlab-Spin-Flat-MicroDuck \
  --checkpoint-file "$CKPT" --onnx-file "$RUN_DIR/spin_baked.onnx" 2>&1 | tail -2

# Digital twin: GIF + metrics (25 fps, 1 chu kỳ envelope 4 s = ~361°)
export MUJOCO_GL=egl
uv run python -m mjlab_microduck.rtx.digital_twin --key spin

### Bước 7: Chuyển logs + model về Drive & tải ONNX về máy

Đủ bộ để mang về máy local: checkpoints, ONNX baked, GIF + metrics digital
twin, wandb offline run (sync sau bằng `uv run wandb sync <dir>` nếu muốn).

In [ ]:
import glob, os, shutil

run_dir = os.path.dirname(sorted(glob.glob('logs/rsl_rl/spin/*/model_*.pt'), key=os.path.getmtime)[-1])
stamp = os.path.basename(run_dir)

# 1) Toàn bộ run (checkpoints + params + tfevents + onnx)
final_dir = f'{DRIVE_DIR}/final_{stamp}'
os.makedirs(final_dir, exist_ok=True)
for f in glob.glob(os.path.join(run_dir, '*')):
    if os.path.isfile(f):
        shutil.copy2(f, final_dir)

# 2) Digital twin (GIF + metrics)
for f in glob.glob('logs/digital_twin/spin/*'):
    shutil.copy2(f, final_dir)

# 3) wandb offline run (nếu có)
for wb in glob.glob('wandb/offline-run-*'):
    shutil.copytree(wb, f'{final_dir}/{os.path.basename(wb)}')

onnx = sorted(glob.glob(f'{final_dir}/*.onnx'), key=os.path.getmtime)[-1]
print('✓ Đã lưu vào Drive:', final_dir)
print('✓ ONNX:', onnx)

from google.colab import files
files.download(onnx)

---
### Sau khi phiên Colab kết thúc

Về máy local (thư mục repo), đồng bộ kết quả từ Drive:

```bash
mkdir -p logs/rsl_rl/spin
cp -r "/path/GoogleDrive/MyDrive/Microduck_RTX/spin/final_"*/* logs/rsl_rl/spin/
uv run rtx-pipeline scan        # matrix sẽ hiện spin READY + twin ✓
uv run rtx-dashboard            # xem model mới trong RTX Control Center
```